# Analiza i Modyfikacja Histogramu Obrazu (Contrast Enhancement)

Niniejszy notatnik prezentuje zaawansowane metody analizy i korekcji kontrastu obrazów cyfrowych. Opiera się na modyfikacji rozkładu gęstości prawdopodobieństwa wystąpienia poszczególnych poziomów jasności (histogramu).

Projekt obejmuje:
- Ekstrakcję i wizualizację histogramów dla obrazów w skali szarości.
- Liniowe rozciąganie histogramu (Contrast Stretching) maksymalizujące wykorzystanie zakresu dynamiki sensora.
- Nieliniowe, globalne wyrównywanie histogramu (Histogram Equalization) z wykorzystaniem dystrybuanty (CDF).
- Lokalną, adaptacyjną poprawę kontrastu algorytmem **CLAHE** (Contrast Limited Adaptive Histogram Equalization) w celu redukcji wzmocnienia szumów.
- Analizę i optymalizację poprawy kontrastu w obrazach kolorowych poprzez transformację z domyślnej przestrzeni **RGB** do percepcyjnej przestrzeni **HSV** (modulacja wyłącznie kanału luminancji - V).

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

if not os.path.exists("lena1.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/lena1.bmp --no-check-certificate

if not os.path.exists("lena2.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/lena2.bmp --no-check-certificate

if not os.path.exists("lena3.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/lena3.bmp --no-check-certificate

if not os.path.exists("lena4.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/lena4.bmp --no-check-certificate

img1 = cv2.imread("lena1.bmp",cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("lena2.bmp",cv2.IMREAD_GRAYSCALE)
img3 = cv2.imread("lena3.bmp",cv2.IMREAD_GRAYSCALE)
img4 = cv2.imread("lena4.bmp",cv2.IMREAD_GRAYSCALE)
his1 = cv2.calcHist(img1,[0],None,[256],[0,256])
his2 = cv2.calcHist(img2,[0],None,[256],[0,256])
his3 = cv2.calcHist(img3,[0],None,[256],[0,256])
his4 = cv2.calcHist(img4,[0],None,[256],[0,256])
figLena, axsLena = plt.subplots(2, 4)

figLena.set_size_inches(20, 10)
plt.subplot(2, 4, 1)
axsLena[0, 0].imshow(img1, 'gray', vmin=0, vmax=256)
axsLena[0, 0].axis('off')
axsLena[1, 0].plot(his1)
axsLena[1, 0].grid()

plt.subplot(2, 4, 2)
axsLena[0, 1].imshow(img2, 'gray', vmin=0, vmax=256)
axsLena[0, 1].axis('off')
axsLena[1, 1].plot(his2)
axsLena[1, 1].grid()

plt.subplot(2, 4, 3)
axsLena[0, 2].imshow(img3, 'gray', vmin=0, vmax=256)
axsLena[0, 2].axis('off')
axsLena[1, 2].plot(his3)
axsLena[1, 2].grid()     
   
plt.subplot(2, 4, 4)    
axsLena[0, 3].imshow(img4, 'gray', vmin=0, vmax=256)
axsLena[0, 3].axis('off')
axsLena[1, 3].plot(his4)
axsLena[1, 3].grid()
plt.show()


## Liniowe Rozciąganie Histogramu (Contrast Stretching)
W przypadku wąskiego histogramu (gdzie obraz jest mało kontrastowy i "szary"), stosuje się operację min-max normalizacji. Pozwala to na przeskalowanie ekstremów jasności tak, aby pokrywały cały dostępny w formacie 8-bitowym zakres od 0 do 255.

In [ ]:
if not os.path.exists("hist1.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/hist1.bmp --no-check-certificate

if not os.path.exists("hist2.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/hist2.bmp --no-check-certificate

if not os.path.exists("hist3.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/hist3.bmp --no-check-certificate

if not os.path.exists("hist4.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/hist4.bmp --no-check-certificate

hist1 = cv2.imread("hist1.bmp",cv2.IMREAD_GRAYSCALE)
hist1n = cv2.normalize(hist1, np.zeros(hist1.shape,np.uint8), 0, 255, cv2.NORM_MINMAX)
hist1h = cv2.calcHist([hist1],[0],None,[256],[0,256])

hist2 = cv2.imread("hist2.bmp",cv2.IMREAD_GRAYSCALE)
hist2n = cv2.normalize(hist2, np.zeros(hist2.shape,np.uint8), 0, 255, cv2.NORM_MINMAX)
hist2h = cv2.calcHist([hist2],[0],None,[256],[0,256])

hist3 = cv2.imread("hist3.bmp",cv2.IMREAD_GRAYSCALE)
hist3n = cv2.normalize(hist3, np.zeros(hist3.shape,np.uint8), 0, 255, cv2.NORM_MINMAX)
hist3h = cv2.calcHist([hist3],[0],None,[256],[0,256])

hist4 = cv2.imread("hist4.bmp",cv2.IMREAD_GRAYSCALE)
hist4n = cv2.normalize(hist4, np.zeros(hist4.shape,np.uint8), 0, 255, cv2.NORM_MINMAX)
hist4h = cv2.calcHist([hist4],[0],None,[256],[0,256])


figHist, axsHist = plt.subplots(2, 4)
figHist.set_size_inches(20, 10)
plt.subplot(2, 4, 1)
axsHist[0, 0].imshow(hist1n, 'gray', vmin=0, vmax=256)
axsHist[0, 0].axis('off')
axsHist[1, 0].plot(hist1h)
axsHist[1, 0].grid()                
plt.subplot(2, 4, 2)
axsHist[0, 1].imshow(hist2n, 'gray', vmin=0, vmax=256)
axsHist[0, 1].axis('off')
axsHist[1, 1].plot(hist2h)
axsHist[1, 1].grid()        
plt.subplot(2, 4, 3)
axsHist[0, 2].imshow(hist3n, 'gray', vmin=0, vmax=256)
axsHist[0, 2].axis('off')
axsHist[1, 2].plot(hist3h)
axsHist[1, 2].grid()        
plt.subplot(2, 4, 4)
axsHist[0, 3].imshow(hist4n, 'gray', vmin=0, vmax=256)
axsHist[0, 3].axis('off')
axsHist[1, 3].plot(hist4h)
axsHist[1, 3].grid()
plt.show()


## Nieliniowe Wyrównywanie Histogramu i Metoda CLAHE
Gdy rozciąganie liniowe nie przynosi pożądanych rezultatów (np. ze względu na obecność bardzo jasnych lub ciemnych szumów), stosuje się nieliniowe transformacje wykorzystujące dystrybuantę (Cumulative Distribution Function - CDF).

Globalne wyrównanie histogramu (HE - Histogram Equalization) może prowadzić do przesterowań i nienaturalnego wyglądu jednolitych płaszczyzn. Problem ten skutecznie rozwiązuje metoda **CLAHE (Contrast Limited Adaptive Histogram Equalization)**, która przetwarza obraz iterując po małych, lokalnych fragmentach (kafelkach) z jednoczesnym nałożeniem limitu na maksymalny przyrost kontrastu.

In [ ]:
hist1_cumsum = hist1h.cumsum()
hist1_cumsum_scaled = hist1_cumsum * hist1h.max() / hist1_cumsum.max()

figHistCum, axsHistCum = plt.subplots()
axsHistCum.plot(hist1_cumsum_scaled)
axsHistCum.plot(hist1h)
axsHistCum.grid()
plt.show()

masked = np.ma.masked_equal(hist1_cumsum, 0)
masked = (masked - masked.min()) * 255 / (masked.max() - masked.min())
lut = np.ma.filled(masked, 0).astype('uint8')

hist1_eq = cv2.LUT(hist1, lut)
hist1_eq_h = cv2.calcHist([hist1_eq], [0], None, [256], [0, 256])
hist1_eq_c = hist1_eq_h.cumsum()
hist1_eq_c = hist1_eq_c * hist1_eq_h.max() / hist1_eq_c.max()

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(10, 5)
axs[0].imshow(hist1_eq, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].plot(hist1_eq_h)
axs[1].plot(hist1_eq_c)
axs[1].grid()
plt.show()

hist1_cv = cv2.equalizeHist(hist1)
hist1_cv_h = cv2.calcHist([hist1_cv], [0], None, [256], [0, 256])
hist1_cv_c = hist1_cv_h.cumsum()
hist1_cv_c = hist1_cv_c * hist1_cv_h.max() / hist1_cv_c.max()

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(10, 5)
axs[0].imshow(hist1_cv, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].plot(hist1_cv_h)
axs[1].plot(hist1_cv_c)
axs[1].grid()
plt.show()

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
hist1_clahe = clahe.apply(hist1)
hist1_clahe_h = cv2.calcHist([hist1_clahe], [0], None, [256], [0, 256])
hist1_clahe_c = hist1_clahe_h.cumsum()
hist1_clahe_c = hist1_clahe_c * hist1_clahe_h.max() / hist1_clahe_c.max()

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(10, 5)
axs[0].imshow(hist1_clahe, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].plot(hist1_clahe_h)
axs[1].plot(hist1_clahe_c)
axs[1].grid()
plt.show()


hist2_eq = cv2.equalizeHist(hist2)
hist2_clahe = clahe.apply(hist2)

fig, axs = plt.subplots(1, 4)
fig.set_size_inches(20, 5)
axs[0].imshow(hist2, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].imshow(hist2n, 'gray', vmin=0, vmax=256)
axs[1].axis('off')
axs[2].imshow(hist2_eq, 'gray', vmin=0, vmax=256)
axs[2].axis('off')
axs[3].imshow(hist2_clahe, 'gray', vmin=0, vmax=256)
axs[3].axis('off')
plt.show()


hist3_eq = cv2.equalizeHist(hist3)
hist3_clahe = clahe.apply(hist3)

fig, axs = plt.subplots(1, 4)
fig.set_size_inches(20, 5)
axs[0].imshow(hist3, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].imshow(hist3n, 'gray', vmin=0, vmax=256)
axs[1].axis('off')
axs[2].imshow(hist3_eq, 'gray', vmin=0, vmax=256)
axs[2].axis('off')
axs[3].imshow(hist3_clahe, 'gray', vmin=0, vmax=256)
axs[3].axis('off')
plt.show()

hist4_eq = cv2.equalizeHist(hist4)
hist4_clahe = clahe.apply(hist4)

fig, axs = plt.subplots(1, 4)
fig.set_size_inches(20, 5)
axs[0].imshow(hist4, 'gray', vmin=0, vmax=256)
axs[0].axis('off')
axs[1].imshow(hist4n, 'gray', vmin=0, vmax=256)
axs[1].axis('off')
axs[2].imshow(hist4_eq, 'gray', vmin=0, vmax=256)
axs[2].axis('off')
axs[3].imshow(hist4_clahe, 'gray', vmin=0, vmax=256)
axs[3].axis('off')
plt.show()

## Zarządzanie kontrastem w przestrzeniach kolorowych (RGB vs HSV)
Naiwne wyrównywanie histogramu niezależnie dla każdego kanału w domyślnej przestrzeni BGR/RGB prowadzi do **katastrofalnych zaburzeń tonalnych** (całkowita zmiana odcienia barw wynikająca z rozerwania proporcji między składowymi czerwieni, zieleni i błękitu).

Aby poprawić jakość i kontrast zdjęć bez wprowadzania przebarwień, konieczna jest konwersja do przestrzeni percepcyjnych (np. **HSV** - Hue, Saturation, Value) i aplikacja wyrównywania histogramu **wyłącznie na kanale luminancji (jasności)**. Ostatnim krokiem jest ponowna, bezstratna konwersja do przestrzeni wyświetlania (RGB).

In [ ]:
if not os.path.exists("lenaRGB.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/lenaRGB.bmp --no-check-certificate

if not os.path.exists("jezioro.jpg") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/03_Histogram/jezioro.jpg --no-check-certificate

def func(img_path):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Ewaluacja niezależnych kanałów RGB
    hist_r = cv2.calcHist([img_rgb], [0], None, [256], [0, 256])
    hist_g = cv2.calcHist([img_rgb], [1], None, [256], [0, 256])
    hist_b = cv2.calcHist([img_rgb], [2], None, [256], [0, 256])

    fig, axs = plt.subplots(1, 4)
    fig.set_size_inches(20, 5)
    axs[0].imshow(img_rgb)
    axs[0].axis('off')
    axs[1].plot(hist_r), axs[1].set_title("Kanał R")
    axs[2].plot(hist_g), axs[2].set_title("Kanał G")
    axs[3].plot(hist_b), axs[3].set_title("Kanał B")
    plt.show()

    # Naiwne wyrównywanie w przestrzeni RGB (zaburzenie kolorów)
    img_rgb_eq = np.zeros(img_rgb.shape, dtype=np.uint8)
    img_rgb_eq[:, :, 0] = cv2.equalizeHist(img_rgb[:, :, 0])
    img_rgb_eq[:, :, 1] = cv2.equalizeHist(img_rgb[:, :, 1])
    img_rgb_eq[:, :, 2] = cv2.equalizeHist(img_rgb[:, :, 2])

    fig, axs = plt.subplots(1, 2)
    fig.set_size_inches(10, 5)
    axs[0].imshow(img_rgb), axs[0].set_title("Oryginał")
    axs[0].axis('off')
    axs[1].imshow(img_rgb_eq), axs[1].set_title("Naiwne wyrównywanie RGB (Błędne)")
    axs[1].axis('off')
    plt.show()

    # Transformacja do HSV i separacja luminancji
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    hist_h = cv2.calcHist([img_hsv], [0], None, [256], [0, 256])
    hist_s = cv2.calcHist([img_hsv], [1], None, [256], [0, 256])
    hist_v = cv2.calcHist([img_hsv], [2], None, [256], [0, 256])

    fig, axs = plt.subplots(1, 3)
    fig.set_size_inches(15, 5)
    axs[0].plot(hist_h), axs[0].set_title("Kanał H (Barwa)")
    axs[1].plot(hist_s), axs[1].set_title("Kanał S (Nasycenie)")
    axs[2].plot(hist_v), axs[2].set_title("Kanał V (Jasność)")
    plt.show()

    # Właściwe wyrównywanie wyłącznie na kanale jasności (V)
    img_hsv[:, :, 2] = cv2.equalizeHist(img_hsv[:, :, 2])
    img_hsv_rgb_eq = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)

    fig, axs = plt.subplots(1, 2)
    fig.set_size_inches(10, 5)
    axs[0].imshow(img_rgb), axs[0].set_title("Oryginał")
    axs[0].axis('off')
    axs[1].imshow(img_hsv_rgb_eq), axs[1].set_title("Właściwa korekcja z zachowaniem barw (HSV)")
    axs[1].axis('off')
    plt.show()

func("lenaRGB.bmp")
func("jezioro.jpg")


### Czyszczenie plików tymczasowych
Aby zapobiec zaśmiecaniu folderu roboczego i repozytorium GitHub, poniższy skrypt automatycznie usuwa obrazy surowe pobrane za pośrednictwem modułu `wget` w trakcie działania notatnika.

In [ ]:
import os
import glob

trash_files = glob.glob('*.bmp*') + glob.glob('*.jpg*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Sprzątanie zakończone! Przestrzeń robocza oczyszczona.")